# Geo-Aware MRO Inventory Intelligence
#
# End-to-end inventory classification and optimization pipeline.
#
# Concepts:
# - ABC classification
# - VED criticality
# - FNS demand behavior
# - Location operational risk
# - Lead-time risk
# - Composite Criticality Index (Ci)
# - Newsvendor optimization
#
# Stack:
# - DVC
# - MLflow
# - Pytest
# - DuckDB

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.pipelines.sku_pipeline import run_pipeline

In [ ]:
df = run_pipeline(n_skus=500)

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

df["abc_class"].value_counts().plot(
    kind="bar",
    ax=ax
)

ax.set_title("ABC Distribution")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

df["ved_class"].value_counts().plot(
    kind="bar",
    ax=ax
)

ax.set_title("VED Distribution")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

df["fns_class"].value_counts().plot(
    kind="bar",
    ax=ax
)

ax.set_title("FNS Distribution")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

df["ci_score"].plot(
    kind="hist",
    bins=20,
    ax=ax
)

ax.set_title("Criticality Index Distribution")

plt.show()

In [ ]:
df[[
    "ci_score",
    "ci_tier",
    "tsl",
    "q_star",
    "rop"
]].head(10)

In [ ]:
top = df.sort_values(
    by="ci_score",
    ascending=False
)

top[[
    "item_id",
    "abc_class",
    "ved_class",
    "fns_class",
    "ci_score",
    "q_star",
    "rop"
]].head(10)

In [ ]:
print("Total SKUs:", len(df))

print(
    "Mean Criticality:",
    round(df["ci_score"].mean(), 4)
)

print(
    "Mean TSL:",
    round(df["tsl"].mean(), 4)
)

print(
    "Mean Q*:",
    round(df["q_star"].mean(), 2)
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pivot = pd.pivot_table(
    df,
    values="ci_score",
    index="abc_class",
    columns="ved_class",
    aggfunc="mean"
)

fig, ax = plt.subplots(figsize=(6, 4))

im = ax.imshow(pivot.values)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)

ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(
            j,
            i,
            round(pivot.values[i, j], 2),
            ha="center",
            va="center",
        )

ax.set_title("Mean Ci Score Heatmap")

plt.colorbar(im)
plt.show()

In [ ]:
pareto = df.sort_values(
    by="annual_consumption_value",
    ascending=False
).reset_index(drop=True)

pareto["cum_acv"] = (
    pareto["annual_consumption_value"]
    .cumsum()
)

pareto["cum_pct"] = (
    pareto["cum_acv"]
    / pareto["annual_consumption_value"].sum()
)

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(
    pareto.index,
    pareto["cum_pct"]
)

ax.set_title("Pareto Curve — ACV Concentration")
ax.set_ylabel("Cumulative %")

plt.show()